# 02 — Planejamento do levantamento cinemático

**Entrada:** requisitos aprovados. **Saída:** plano de campo e de processamento pronto para execução.

> Substitua os parâmetros didáticos pelos valores confirmados no reconhecimento e na documentação vigente do SLAM 100.

## Objetivos de aprendizagem

1. desenhar trajetos com cobertura, redundância e fechamento de loop;
2. estimar tempo, armazenamento e densidade relativa de amostragem;
3. planejar verificações, calibração e georreferenciamento;
4. definir a estrutura de dados, o pipeline e os formatos de entrega;
5. estabelecer critérios de abortar, repetir ou complementar uma campanha.

## 1. Parâmetros e premissas

A trajetória deve começar em uma área geometricamente rica, manter movimento suave, revisitar trechos reconhecíveis e terminar com um loop. Corredores longos, superfícies repetitivas, vidro, multidões e movimento brusco elevam o risco de deriva.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PARAMETROS = {
    'comprimento_rota_m': 850.0,
    'velocidade_m_s': 0.8,
    'taxa_pontos_s': 200_000,  # confirmar na configuração de campo
    'bytes_por_ponto': 32,     # hipótese para estimativa, não tamanho garantido
    'margem_tempo': 1.30,
    'margem_dados': 1.50,
}
PARAMETROS

### 1.1 Estimativas operacionais

A densidade pontual real depende da distância, ângulo de incidência, oclusão, velocidade e padrão do sensor. A conta abaixo estima volume e duração; ela não promete densidade por metro quadrado.

In [ ]:
duracao_s = PARAMETROS['comprimento_rota_m'] / PARAMETROS['velocidade_m_s']
pontos_estimados = duracao_s * PARAMETROS['taxa_pontos_s']
armazenamento_gb = pontos_estimados * PARAMETROS['bytes_por_ponto'] / 1e9
estimativa = pd.Series({
    'duração com margem (min)': duracao_s * PARAMETROS['margem_tempo'] / 60,
    'pontos brutos (milhões)': pontos_estimados / 1e6,
    'armazenamento com margem (GB)': armazenamento_gb * PARAMETROS['margem_dados'],
})
estimativa.round(2).to_frame('estimativa')

## 2. Trajeto, cobertura e loops

O exemplo representa um circuito fechado. Em campo, marque início/fim, cruzamentos, mudanças de pavimento, pontos de controle, zonas críticas e rotas alternativas. Planeje passagens laterais ou em sentido oposto para reduzir oclusões, sem criar dados redundantes sem propósito.

In [ ]:
rota = np.array([[0,0], [40,0], [40,25], [10,25], [10,10], [0,10], [0,0]], dtype=float)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rota[:,0], rota[:,1], '-o', label='trajeto planejado')
ax.scatter(*rota[0], s=120, marker='*', label='início/fim')
ax.set(title='Croqui didático do trajeto', xlabel='Leste local (m)', ylabel='Norte local (m)', aspect='equal')
ax.grid(alpha=.3); ax.legend(); plt.show()

## 3. Calibração, apoio e controle

### Antes da campanha

- sincronize relógios e confira data/hora/fuso;
- inspecione lente, janela do LiDAR, fixações, baterias e armazenamento;
- registre firmware, aplicativo, perfil e operador;
- execute a inicialização/calibração recomendada pelo fabricante;
- faça um trecho curto, processe-o e confirme integridade antes da missão;
- distribua alvos em 3D, evitando pontos colineares ou todos no mesmo plano.

### Georreferenciamento

Defina CRS, época, modelo geoidal/referencial vertical, método GNSS/topográfico, qualidade das coordenadas e transformação. Separe **pontos de controle** (ajuste) de **pontos de verificação** (avaliação independente).

In [ ]:
plano_controle = pd.DataFrame([
    ('PC-01', 'controle', 'entrada', 'A DEFINIR', 'A DEFINIR'),
    ('PC-02', 'controle', 'extremo oposto', 'A DEFINIR', 'A DEFINIR'),
    ('PV-01', 'verificação', 'zona central', 'A DEFINIR', 'A DEFINIR'),
    ('PV-02', 'verificação', 'mudança de cota', 'A DEFINIR', 'A DEFINIR'),
], columns=['id', 'uso', 'local', 'coordenada', 'incerteza'])
plano_controle

## 4. Plano de processamento e dados

Fluxo proposto: **ingestão → cópia imutável → inventário/hashes → processamento do fabricante → registro/SLAM → georreferenciamento → limpeza → classificação → controle de qualidade → produtos**.

Estrutura sugerida:

```text
dados/00_brutos/campanha_01/
dados/01_intermediarios/
dados/02_processados/
dados/03_controle/
produtos/
relatorios/
```

- **LAS/LAZ:** entrega geoespacial e classificação;
- **E57:** intercâmbio de varreduras, imagens e poses;
- **PLY:** visualização, malhas e protótipos;
- **CSV/GeoPackage:** controles, trajetórias, feições e metadados tabulares.

PDAL é adequado para pipelines geoespaciais repetíveis; Open3D, para registro, geometria, visualização e reconstrução. Documente versões e parâmetros.

## 5. Checklist de liberação

- [ ] rota principal e alternativa reconhecidas;
- [ ] loops, transições, oclusões e pontos de apoio marcados;
- [ ] autonomia e armazenamento calculados com margem;
- [ ] equipe, EPI, comunicações, acesso e previsão confirmados;
- [ ] convenção de nomes, estrutura de pastas e cópia 3-2-1 definidas;
- [ ] critérios de parada, repetição e campanha complementar registrados;
- [ ] pipeline e formatos compatíveis com os critérios de aceite.

**Entrega do módulo:** plano de levantamento, croqui da rota, plano de apoio/controle e plano de processamento.

**Próximo módulo:** [03 — Execução](03_Execucao.ipynb).